In [0]:
"""
id: source_a3f7b2e1
template: sql
templateVersion: 1.0.0
name: Product PDFs
position:
  x: 0
  y: 0
description:
  text: Read all PDF files from the specified folder.
  hash: 60d1d3be
previewCodeHash: 4c06c6e8106aa8a2
previewMode: "1000"
config:
  query: |-
    SELECT *
    FROM read_files(
      'dbfs:/Volumes/fevm_master_classic_marcus_catalog/rfp_presentation/product_pdfs',
      format => 'binaryFile',
      pathGlobFilter => '*.pdf'
    )
input: []
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "query": "SELECT *\nFROM read_files(\n  'dbfs:/Volumes/fevm_master_classic_marcus_catalog/rfp_presentation/product_pdfs',\n  format => 'binaryFile',\n  pathGlobFilter => '*.pdf'\n)"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_a3f7b2e1.result"] = out["result"]
if globals().get("ld_display_outputs", False) or "source_a3f7b2e1" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["source_a3f7b2e1.result"])

In [0]:
"""
id: ai_function_c4d8e9f2
template: ai_function
templateVersion: 3.0.0
name: Parse PDFs
position:
  x: 260
  y: 0
description:
  text: Extract product code from file path, parse document content, and add file details without keeping original columns.
  hash: 216fbb3f
previewCodeHash: d5285fa72cd6716c
previewMode: "1000"
config:
  expressions:
    - path AS `file_path`
    - regexp_extract(path, '([^/]+)\\.[Pp][Dd][Ff]$', 1) AS `product_code`
    - ai_parse_document(content, MAP('version', '2.0')) AS `parsed_content`
    - length(content) AS `file_size_bytes`
    - current_timestamp() AS `parsed_at`
  keep_all_columns: false
input:
  - node: source_a3f7b2e1
    input_port: data
    output_port: result
"""

# generated from the system
from typing import Dict, Any, List
import hashlib

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]

    # Table-valued AI functions (ai_forecast, ...) live in the FROM
    # clause and have no SELECT-list shape, so we run them via
    # spark.sql and bind the upstream DataFrame as a session-scoped
    # temp view substituted in for the
    # __lakebuilder_ai_function_input__ placeholder identifier
    # (which is a real SQL identifier so the persisted statement
    # round-trips through the SQL parser when the cell reloads).
    #
    # spark.sql returns a lazy DataFrame: the view name is resolved
    # by Spark at action time (preview limit/collect), not when
    # spark.sql is called. We therefore deliberately do NOT drop
    # the temp view here — dropping it would leave the lazy plan
    # pointing at a missing relation and trigger TABLE_OR_VIEW_NOT
    # _FOUND when the downstream action fires.
    #
    # The view name is derived deterministically from (tvf_sql,
    # id(df)) so reruns of the same cell reuse the same name and
    # createOrReplaceTempView keeps the session catalog bounded at
    # one entry per (cell × upstream) instead of growing one entry
    # per run. id(df) is included to keep two cells that happen to
    # have identical tvf_sql but distinct upstream DataFrames from
    # clobbering each other's bindings.
    tvf_sql: str = config.get("tvf_sql") or ""
    if tvf_sql:
        key = f"{tvf_sql}\x00{id(df)}".encode("utf-8")
        view_name = f"lakebuilder_ai_fn_{hashlib.sha256(key).hexdigest()[:12]}"
        df.createOrReplaceTempView(view_name)
        sql = tvf_sql.replace("__lakebuilder_ai_function_input__", view_name)
        return {"ai_data": spark.sql(sql)}

    expressions: List[str] = config.get("expressions", [])
    if not expressions:
        return {"ai_data": df}

    keep_all_columns: bool = config.get("keep_all_columns", True)
    if keep_all_columns:
        return {"ai_data": df.selectExpr(*expressions, "*")}
    return {"ai_data": df.selectExpr(*expressions)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "path AS `file_path`",
        "regexp_extract(path, '([^/]+)\\\\.[Pp][Dd][Ff]$', 1) AS `product_code`",
        "ai_parse_document(content, MAP('version', '2.0')) AS `parsed_content`",
        "length(content) AS `file_size_bytes`",
        "current_timestamp() AS `parsed_at`"
    ],
    "keep_all_columns": False
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["source_a3f7b2e1.result"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["ai_function_c4d8e9f2.ai_data"] = out["ai_data"]
if globals().get("ld_display_outputs", False) or "ai_function_c4d8e9f2" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["ai_function_c4d8e9f2.ai_data"])

In [0]:
"""
id: ai_function_b7a6c5d3
template: ai_function
templateVersion: 3.0.0
name: Classify brochures
position:
  x: 520
  y: 0
description:
  text: Add product classification based on parsed content, keeping all original columns.
  hash: bcf577f3
previewCodeHash: 9b507e54182c90c8
previewMode: "1000"
config:
  expressions:
    - "ai_classify(parsed_content, '{\"CREDIT_CARD\": \"Credit card products\", \"PERSONAL_LOAN\": \"Personal loan products\", \"HOME_LOAN\": \"Home loan or mortgage products\", \"INVESTMENT\": \"Investment or fund products\", \"INSURANCE\": \"Insurance products\"}', MAP('version', '2.1')) AS `classification`"
  keep_all_columns: true
input:
  - node: ai_function_c4d8e9f2
    input_port: data
    output_port: ai_data
"""

# generated from the system
from typing import Dict, Any, List
import hashlib

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]

    # Table-valued AI functions (ai_forecast, ...) live in the FROM
    # clause and have no SELECT-list shape, so we run them via
    # spark.sql and bind the upstream DataFrame as a session-scoped
    # temp view substituted in for the
    # __lakebuilder_ai_function_input__ placeholder identifier
    # (which is a real SQL identifier so the persisted statement
    # round-trips through the SQL parser when the cell reloads).
    #
    # spark.sql returns a lazy DataFrame: the view name is resolved
    # by Spark at action time (preview limit/collect), not when
    # spark.sql is called. We therefore deliberately do NOT drop
    # the temp view here — dropping it would leave the lazy plan
    # pointing at a missing relation and trigger TABLE_OR_VIEW_NOT
    # _FOUND when the downstream action fires.
    #
    # The view name is derived deterministically from (tvf_sql,
    # id(df)) so reruns of the same cell reuse the same name and
    # createOrReplaceTempView keeps the session catalog bounded at
    # one entry per (cell × upstream) instead of growing one entry
    # per run. id(df) is included to keep two cells that happen to
    # have identical tvf_sql but distinct upstream DataFrames from
    # clobbering each other's bindings.
    tvf_sql: str = config.get("tvf_sql") or ""
    if tvf_sql:
        key = f"{tvf_sql}\x00{id(df)}".encode("utf-8")
        view_name = f"lakebuilder_ai_fn_{hashlib.sha256(key).hexdigest()[:12]}"
        df.createOrReplaceTempView(view_name)
        sql = tvf_sql.replace("__lakebuilder_ai_function_input__", view_name)
        return {"ai_data": spark.sql(sql)}

    expressions: List[str] = config.get("expressions", [])
    if not expressions:
        return {"ai_data": df}

    keep_all_columns: bool = config.get("keep_all_columns", True)
    if keep_all_columns:
        return {"ai_data": df.selectExpr(*expressions, "*")}
    return {"ai_data": df.selectExpr(*expressions)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "ai_classify(parsed_content, '{\"CREDIT_CARD\": \"Credit card products\", \"PERSONAL_LOAN\": \"Personal loan products\", \"HOME_LOAN\": \"Home loan or mortgage products\", \"INVESTMENT\": \"Investment or fund products\", \"INSURANCE\": \"Insurance products\"}', MAP('version', '2.1')) AS `classification`"
    ],
    "keep_all_columns": True
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["ai_function_c4d8e9f2.ai_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["ai_function_b7a6c5d3.ai_data"] = out["ai_data"]
if globals().get("ld_display_outputs", False) or "ai_function_b7a6c5d3" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["ai_function_b7a6c5d3.ai_data"])

In [0]:
"""
id: prepare_e1f2a3b4
template: prepare
templateVersion: 1.0.0
name: Extract category
position:
  x: 780
  y: 0
description:
  text: Create a new column product_category with values extracted from classification response array.
  hash: c8a8bc0f
previewCodeHash: ee222b27bdc29f5c
previewMode: "1000"
config:
  actions:
    - type: formula
      target: product_category
      expression: classification:response[0].value::STRING
input:
  - node: ai_function_b7a6c5d3
    input_port: data
    output_port: ai_data
"""

# generated from the system
from typing import Any, Callable, Dict, List
import pyspark.sql.functions as F

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Wrap a user-provided value as a literal cast to the target column's type.

    UI always feeds text per the operator spec; this is where the
    type coercion happens so the per-action UI stays simple.
    """
    return F.lit(value).cast(target_dtype)

def _apply_formula(df, action: Dict[str, Any]):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    return df.withColumn(target, F.expr(expression))

def _apply_cast(df, action: Dict[str, Any]):
    column = action.get("column", "")
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "cast")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return df.withColumn(
            column,
            F.expr(f"try_cast({_quoted_ident(column)} as {to_type})"),
        )
    return df.withColumn(column, _col(column).cast(to_type))

def _apply_replace_value(df, action: Dict[str, Any]):
    column = action.get("column", "")
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    _require_column(df, column, "replace_value")
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(predicate, replacement).otherwise(_col(column)),
    )

def _apply_fill_null(df, action: Dict[str, Any]):
    column = action.get("column", "")
    with_val = action.get("with", "")
    _require_column(df, column, "fill_null")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(_col(column).isNull(), replacement).otherwise(_col(column)),
    )

def _apply_text_case(df, action: Dict[str, Any]):
    column = action.get("column", "")
    case = action.get("case", "")
    _require_column(df, column, "text_case")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_trim(df, action: Dict[str, Any]):
    column = action.get("column", "")
    side = action.get("side", "both")
    _require_column(df, column, "trim")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_regex_replace(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    _require_column(df, column, "regex_replace")
    return df.withColumn(
        column,
        F.regexp_replace(_col(column), pattern, replacement),
    )

def _apply_extract(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    target = action.get("target") or column
    group_raw = action.get("group", 0)
    _require_column(df, column, "extract")
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return df.withColumn(
        target,
        F.regexp_extract(_col(column), pattern, group_idx),
    )

def _apply_parse_date(df, action: Dict[str, Any]):
    column = action.get("column", "")
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_date")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    # PySpark exposes F.try_to_timestamp from Spark 3.5 but only adds
    # F.try_to_date in Spark 4.0. Current Databricks Runtimes still ship
    # Spark 3.5, where referencing F.try_to_date raises AttributeError
    # before any data is touched. Route the null-on-error path through
    # primitives that exist in both 3.5 and 4.0:
    #   - F.try_to_timestamp  (PySpark 3.5+, returns NULL under ANSI too)
    #   - SQL try_cast        (Spark 3.5+, returns NULL under ANSI too)
    #   - F.to_date           (PySpark 3.5+; returns NULL on parse failure
    #                          under the default non-ANSI mode, which is
    #                          the Databricks default. Under ANSI mode it
    #                          raises — accepted limitation until
    #                          try_to_date lands on every supported DBR.)
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return df.withColumn(
                    column, F.try_to_timestamp(_col(column), F.lit(fmt))
                )
            return df.withColumn(column, F.try_to_timestamp(_col(column)))
        # kind == "date"
        if fmt:
            return df.withColumn(column, F.to_date(_col(column), fmt))
        return df.withColumn(
            column, F.expr(f"try_cast({_quoted_ident(column)} as date)")
        )
    # on_error == "error": surface parse failures as Spark exceptions.
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return df.withColumn(column, fn(_col(column), fmt))
    return df.withColumn(column, fn(_col(column)))

ACTION_DISPATCH: Dict[str, Callable] = {
    "formula": _apply_formula,
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        fn = ACTION_DISPATCH.get(action_type)
        if fn is None:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {sorted(ACTION_DISPATCH.keys())}"
            )
        df = fn(df, action)
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "target": "product_category",
            "expression": "classification:response[0].value::STRING"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["ai_function_b7a6c5d3.ai_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prepare_e1f2a3b4.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prepare_e1f2a3b4" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prepare_e1f2a3b4.prepared_data"])

In [0]:
"""
id: ai_function_d9e8f7a6
template: ai_function
templateVersion: 3.0.0
name: Extract product fields
position:
  x: 1040
  y: 0
description:
  text: Extract specified financial product information fields while keeping all original columns.
  hash: "29003084"
previewCodeHash: 63aef5624db4ba6e
previewMode: "1000"
config:
  expressions:
    - ai_extract(parsed_content, '{"product_name":{"type":"string"},"min_amount_myr":{"type":"string"},"max_amount_myr":{"type":"string"},"min_rate_pct":{"type":"string"},"max_rate_pct":{"type":"string"},"min_tenure_months":{"type":"string"},"max_tenure_months":{"type":"string"},"min_income_annual_myr":{"type":"string"},"shariah_compliant":{"type":"string"},"annual_fee_myr":{"type":"string"},"annual_fee_waiver":{"type":"string"},"processing_fee":{"type":"string"},"early_termination_fee":{"type":"string"},"key_benefit_1":{"type":"string"},"key_benefit_2":{"type":"string"},"key_benefit_3":{"type":"string"},"fees_summary":{"type":"string"},"eligibility_age_min":{"type":"string"},"eligibility_age_max":{"type":"string"},"eligibility_nationality":{"type":"string"},"eligibility_employment":{"type":"string"},"documents_required":{"type":"string"},"cashback_rate_pct":{"type":"string"},"reward_points_per_rm":{"type":"string"},"lounge_access_included":{"type":"string"},"profit_rate_type":{"type":"string"},"collateral_required":{"type":"string"},"max_dsr_pct":{"type":"string"},"fund_risk_rating":{"type":"string"},"capital_guaranteed":{"type":"string"},"distribution_frequency":{"type":"string"},"sum_covered_max_myr":{"type":"string"},"effective_date":{"type":"string"},"product_code_extracted":{"type":"string"}}', MAP('version', '2.1', 'instructions', 'These are Malaysian financial product brochures. Extract all 34 fields. Return null for fields not applicable to the product type.')) `extracted_fields`
  keep_all_columns: true
input:
  - node: prepare_e1f2a3b4
    input_port: data
    output_port: prepared_data
"""

# generated from the system
from typing import Dict, Any, List
import hashlib

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]

    # Table-valued AI functions (ai_forecast, ...) live in the FROM
    # clause and have no SELECT-list shape, so we run them via
    # spark.sql and bind the upstream DataFrame as a session-scoped
    # temp view substituted in for the
    # __lakebuilder_ai_function_input__ placeholder identifier
    # (which is a real SQL identifier so the persisted statement
    # round-trips through the SQL parser when the cell reloads).
    #
    # spark.sql returns a lazy DataFrame: the view name is resolved
    # by Spark at action time (preview limit/collect), not when
    # spark.sql is called. We therefore deliberately do NOT drop
    # the temp view here — dropping it would leave the lazy plan
    # pointing at a missing relation and trigger TABLE_OR_VIEW_NOT
    # _FOUND when the downstream action fires.
    #
    # The view name is derived deterministically from (tvf_sql,
    # id(df)) so reruns of the same cell reuse the same name and
    # createOrReplaceTempView keeps the session catalog bounded at
    # one entry per (cell × upstream) instead of growing one entry
    # per run. id(df) is included to keep two cells that happen to
    # have identical tvf_sql but distinct upstream DataFrames from
    # clobbering each other's bindings.
    tvf_sql: str = config.get("tvf_sql") or ""
    if tvf_sql:
        key = f"{tvf_sql}\x00{id(df)}".encode("utf-8")
        view_name = f"lakebuilder_ai_fn_{hashlib.sha256(key).hexdigest()[:12]}"
        df.createOrReplaceTempView(view_name)
        sql = tvf_sql.replace("__lakebuilder_ai_function_input__", view_name)
        return {"ai_data": spark.sql(sql)}

    expressions: List[str] = config.get("expressions", [])
    if not expressions:
        return {"ai_data": df}

    keep_all_columns: bool = config.get("keep_all_columns", True)
    if keep_all_columns:
        return {"ai_data": df.selectExpr(*expressions, "*")}
    return {"ai_data": df.selectExpr(*expressions)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "ai_extract(parsed_content, '{\"product_name\":{\"type\":\"string\"},\"min_amount_myr\":{\"type\":\"string\"},\"max_amount_myr\":{\"type\":\"string\"},\"min_rate_pct\":{\"type\":\"string\"},\"max_rate_pct\":{\"type\":\"string\"},\"min_tenure_months\":{\"type\":\"string\"},\"max_tenure_months\":{\"type\":\"string\"},\"min_income_annual_myr\":{\"type\":\"string\"},\"shariah_compliant\":{\"type\":\"string\"},\"annual_fee_myr\":{\"type\":\"string\"},\"annual_fee_waiver\":{\"type\":\"string\"},\"processing_fee\":{\"type\":\"string\"},\"early_termination_fee\":{\"type\":\"string\"},\"key_benefit_1\":{\"type\":\"string\"},\"key_benefit_2\":{\"type\":\"string\"},\"key_benefit_3\":{\"type\":\"string\"},\"fees_summary\":{\"type\":\"string\"},\"eligibility_age_min\":{\"type\":\"string\"},\"eligibility_age_max\":{\"type\":\"string\"},\"eligibility_nationality\":{\"type\":\"string\"},\"eligibility_employment\":{\"type\":\"string\"},\"documents_required\":{\"type\":\"string\"},\"cashback_rate_pct\":{\"type\":\"string\"},\"reward_points_per_rm\":{\"type\":\"string\"},\"lounge_access_included\":{\"type\":\"string\"},\"profit_rate_type\":{\"type\":\"string\"},\"collateral_required\":{\"type\":\"string\"},\"max_dsr_pct\":{\"type\":\"string\"},\"fund_risk_rating\":{\"type\":\"string\"},\"capital_guaranteed\":{\"type\":\"string\"},\"distribution_frequency\":{\"type\":\"string\"},\"sum_covered_max_myr\":{\"type\":\"string\"},\"effective_date\":{\"type\":\"string\"},\"product_code_extracted\":{\"type\":\"string\"}}', MAP('version', '2.1', 'instructions', 'These are Malaysian financial product brochures. Extract all 34 fields. Return null for fields not applicable to the product type.')) `extracted_fields`"
    ],
    "keep_all_columns": True
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["prepare_e1f2a3b4.prepared_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["ai_function_d9e8f7a6.ai_data"] = out["ai_data"]
if globals().get("ld_display_outputs", False) or "ai_function_d9e8f7a6" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["ai_function_d9e8f7a6.ai_data"])

In [0]:
"""
id: prepare_f5e4d3c2
template: prepare
templateVersion: 1.0.0
name: Finalize columns
position:
  x: 1300
  y: 0
description:
  text: Add current timestamp as extracted_at column.
  hash: 40e6b934
previewCodeHash: 79244f2250e38f02
previewMode: "1000"
config:
  actions:
    - type: formula
      target: extracted_at
      expression: current_timestamp()
input:
  - node: ai_function_d9e8f7a6
    input_port: data
    output_port: ai_data
"""

# generated from the system
from typing import Any, Callable, Dict, List
import pyspark.sql.functions as F

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Wrap a user-provided value as a literal cast to the target column's type.

    UI always feeds text per the operator spec; this is where the
    type coercion happens so the per-action UI stays simple.
    """
    return F.lit(value).cast(target_dtype)

def _apply_formula(df, action: Dict[str, Any]):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    return df.withColumn(target, F.expr(expression))

def _apply_cast(df, action: Dict[str, Any]):
    column = action.get("column", "")
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "cast")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return df.withColumn(
            column,
            F.expr(f"try_cast({_quoted_ident(column)} as {to_type})"),
        )
    return df.withColumn(column, _col(column).cast(to_type))

def _apply_replace_value(df, action: Dict[str, Any]):
    column = action.get("column", "")
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    _require_column(df, column, "replace_value")
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(predicate, replacement).otherwise(_col(column)),
    )

def _apply_fill_null(df, action: Dict[str, Any]):
    column = action.get("column", "")
    with_val = action.get("with", "")
    _require_column(df, column, "fill_null")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(_col(column).isNull(), replacement).otherwise(_col(column)),
    )

def _apply_text_case(df, action: Dict[str, Any]):
    column = action.get("column", "")
    case = action.get("case", "")
    _require_column(df, column, "text_case")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_trim(df, action: Dict[str, Any]):
    column = action.get("column", "")
    side = action.get("side", "both")
    _require_column(df, column, "trim")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_regex_replace(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    _require_column(df, column, "regex_replace")
    return df.withColumn(
        column,
        F.regexp_replace(_col(column), pattern, replacement),
    )

def _apply_extract(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    target = action.get("target") or column
    group_raw = action.get("group", 0)
    _require_column(df, column, "extract")
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return df.withColumn(
        target,
        F.regexp_extract(_col(column), pattern, group_idx),
    )

def _apply_parse_date(df, action: Dict[str, Any]):
    column = action.get("column", "")
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_date")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    # PySpark exposes F.try_to_timestamp from Spark 3.5 but only adds
    # F.try_to_date in Spark 4.0. Current Databricks Runtimes still ship
    # Spark 3.5, where referencing F.try_to_date raises AttributeError
    # before any data is touched. Route the null-on-error path through
    # primitives that exist in both 3.5 and 4.0:
    #   - F.try_to_timestamp  (PySpark 3.5+, returns NULL under ANSI too)
    #   - SQL try_cast        (Spark 3.5+, returns NULL under ANSI too)
    #   - F.to_date           (PySpark 3.5+; returns NULL on parse failure
    #                          under the default non-ANSI mode, which is
    #                          the Databricks default. Under ANSI mode it
    #                          raises — accepted limitation until
    #                          try_to_date lands on every supported DBR.)
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return df.withColumn(
                    column, F.try_to_timestamp(_col(column), F.lit(fmt))
                )
            return df.withColumn(column, F.try_to_timestamp(_col(column)))
        # kind == "date"
        if fmt:
            return df.withColumn(column, F.to_date(_col(column), fmt))
        return df.withColumn(
            column, F.expr(f"try_cast({_quoted_ident(column)} as date)")
        )
    # on_error == "error": surface parse failures as Spark exceptions.
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return df.withColumn(column, fn(_col(column), fmt))
    return df.withColumn(column, fn(_col(column)))

ACTION_DISPATCH: Dict[str, Callable] = {
    "formula": _apply_formula,
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        fn = ACTION_DISPATCH.get(action_type)
        if fn is None:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {sorted(ACTION_DISPATCH.keys())}"
            )
        df = fn(df, action)
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "target": "extracted_at",
            "expression": "current_timestamp()"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["ai_function_d9e8f7a6.ai_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prepare_f5e4d3c2.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prepare_f5e4d3c2" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prepare_f5e4d3c2.prepared_data"])

In [0]:
"""
id: transform_a2b3c4d5
template: transform
templateVersion: 3.0.0
name: Select final columns
position:
  x: 1560
  y: 0
description:
  text: "Select specific columns: file_path, product_code, product_category, parsed_at, extracted_fields, extracted_at."
  hash: 882fe0cd
previewCodeHash: f8b5cc3da4e54a69
previewMode: "1000"
config:
  mode: select
  edits:
    - column: file_path
    - column: product_code
    - column: product_category
    - column: parsed_at
    - column: extracted_fields
    - column: extracted_at
  ordered:
    - file_path
    - product_code
    - product_category
    - parsed_at
    - extracted_fields
    - extracted_at
input:
  - node: prepare_f5e4d3c2
    input_port: data
    output_port: prepared_data
"""

# generated from the system
from typing import Any, Dict, List, Optional
from pyspark.sql import functions as F
from pyspark.sql import types as _spark_types
from pyspark.sql.types import (
    BinaryType,
    BooleanType,
    DateType,
    DecimalType,
    FractionalType,
    IntegerType,
    IntegralType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

_TIMESTAMP_TYPES = tuple(
    t
    for t in (TimestampType, getattr(_spark_types, "TimestampNTZType", None))
    if t is not None
)

def _col_ref(name: str):
    if "." in name and not (name.startswith("`") and name.endswith("`")):
        return F.col("`" + name.replace("`", "``") + "`")
    return F.col(name)

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _column_for(item: Dict[str, Any]):
    col = _col_ref(item.get("column", ""))
    alias = item.get("alias")
    if alias:
        col = col.alias(alias)
    return col

def _passthrough_column(name: str, rename_map: Dict[str, Dict[str, Any]]):
    entry = rename_map.get(name)
    col = _col_ref(name)
    if entry and entry.get("alias"):
        col = col.alias(entry["alias"])
    return col

def _type_category(data_type) -> Optional[str]:
    if isinstance(data_type, BooleanType):
        return "boolean"
    if isinstance(data_type, DecimalType):
        return "decimal"
    if isinstance(data_type, IntegralType):
        return "integer"
    if isinstance(data_type, FractionalType):
        return "float"
    if isinstance(data_type, StringType):
        return "string"
    if isinstance(data_type, DateType):
        return "date"
    if isinstance(data_type, _TIMESTAMP_TYPES):
        return "timestamp"
    if isinstance(data_type, BinaryType):
        return "binary"
    return None

_METADATA_SCHEMA = StructType(
    [
        StructField("Name", StringType(), False),
        StructField("Type", StringType(), False),
        StructField("Category", StringType(), True),
        StructField("FieldNumber", IntegerType(), False),
        StructField("IsNumeric", BooleanType(), False),
        StructField("IsInteger", BooleanType(), False),
        StructField("IsFloat", BooleanType(), False),
        StructField("IsString", BooleanType(), False),
        StructField("IsDateOrTime", BooleanType(), False),
        StructField("IsBinary", BooleanType(), False),
    ]
)

def _metadata_row(index: int, field):
    data_type = field.dataType
    category = _type_category(data_type)
    return (
        field.name,
        data_type.simpleString(),
        category,
        index + 1,
        category in ("integer", "float", "decimal"),
        isinstance(data_type, IntegralType),
        category == "float",
        isinstance(data_type, StringType),
        isinstance(data_type, (DateType,) + _TIMESTAMP_TYPES),
        isinstance(data_type, BinaryType),
    )

def _sql_string_literal(value: str) -> str:
    return "'" + value.replace("\\", "\\\\").replace("'", "\\'") + "'"

def _predicate_for(dynamic: Dict[str, Any]) -> str:
    kind = dynamic.get("kind")
    if kind == "byType":
        selected = dynamic.get("types") or []
        if not selected:
            return "false"
        quoted = ", ".join(_sql_string_literal(t) for t in selected)
        return "Category IN (" + quoted + ")"
    if kind == "byName":
        pattern = dynamic.get("namePattern")
        if not pattern or not pattern.get("op"):
            raise ValueError(
                "Select: a byName dynamic rule requires namePattern with an 'op' and 'value'"
            )
        op = pattern.get("op")
        value = pattern.get("value", "")
        case_sensitive = pattern.get("caseSensitive", False) is True
        if op == "regex":
            regex = value if case_sensitive else "(?i)" + value
            return "Name rlike " + _sql_string_literal(regex)
        name_expr = "Name" if case_sensitive else "lower(Name)"
        needle = value if case_sensitive else value.lower()
        escaped = needle.replace("\\", "\\\\").replace("%", "\\%").replace("_", "\\_")
        if op == "startsWith":
            like = escaped + "%"
        elif op == "endsWith":
            like = "%" + escaped
        else:
            like = "%" + escaped + "%"
        return name_expr + " like " + _sql_string_literal(like)
    if kind == "byExpression":
        expression = dynamic.get("expression") or ""
        return expression if expression.strip() else "false"
    return "false"

def _resolve_dynamic(df, spark, dynamic: Dict[str, Any]):
    predicate = _predicate_for(dynamic)
    fields = df.schema.fields
    rows = [_metadata_row(i, f) for i, f in enumerate(fields)]
    meta_df = spark.createDataFrame(rows, _METADATA_SCHEMA)
    matched_ordinals = {
        row["FieldNumber"] for row in meta_df.filter(predicate).select("FieldNumber").collect()
    }
    action = dynamic.get("action", "keep")
    return [
        index
        for index in range(len(fields))
        if ((index + 1) in matched_ordinals) != (action == "remove")
    ]

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]

    dynamic = config.get("dynamic")
    if dynamic:
        kept_indexes = _resolve_dynamic(df, spark, dynamic)
        original_names = [field.name for field in df.schema.fields]
        placeholders = ["_c" + str(i) for i in range(len(original_names))]
        projected = df.toDF(*placeholders).select(*(F.col(placeholders[i]) for i in kept_indexes))
        return {
            "transformed_data": projected.toDF(*(original_names[i] for i in kept_indexes))
        }

    mode = config.get("mode", "passthrough")
    edits: List[Dict[str, Any]] = config.get("edits", [])
    ordered: List[str] = config.get("ordered") or []

    if mode == "select":
        checked_edits = [item for item in edits if _is_checked(item)]
        if not checked_edits:
            return {"transformed_data": df}
        order_index = {name: i for i, name in enumerate(ordered)}
        tail = len(order_index)
        ordered_edits = sorted(
            checked_edits,
            key=lambda item: order_index.get(item.get("column", ""), tail),
        )
        return {"transformed_data": df.select(*(_column_for(item) for item in ordered_edits))}

    unchecked_cols = {
        item.get("column", "")
        for item in edits
        if not _is_checked(item)
    }
    rename_map = {
        item.get("column", ""): item
        for item in edits
        if item.get("alias") and _is_checked(item)
    }
    upstream_cols = list(df.columns)
    upstream_set = set(upstream_cols)

    effective_ordered = ordered if ordered else list(upstream_cols)

    placed = set()
    out = []
    for token in effective_ordered:
        if token in placed or token not in upstream_set:
            continue
        placed.add(token)
        if token in unchecked_cols:
            continue
        out.append(_passthrough_column(token, rename_map))

    for col in upstream_cols:
        if col in placed or col in unchecked_cols:
            continue
        out.append(_passthrough_column(col, rename_map))

    if not out:
        return {"transformed_data": df}
    return {"transformed_data": df.select(*out)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "mode": "select",
    "edits": [
        {
            "column": "file_path"
        },
        {
            "column": "product_code"
        },
        {
            "column": "product_category"
        },
        {
            "column": "parsed_at"
        },
        {
            "column": "extracted_fields"
        },
        {
            "column": "extracted_at"
        }
    ],
    "ordered": [
        "file_path",
        "product_code",
        "product_category",
        "parsed_at",
        "extracted_fields",
        "extracted_at"
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["prepare_f5e4d3c2.prepared_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["transform_a2b3c4d5.transformed_data"] = out["transformed_data"]
if globals().get("ld_display_outputs", False) or "transform_a2b3c4d5" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["transform_a2b3c4d5.transformed_data"])

In [0]:
"""
id: output_b1c2d3e4
template: output
templateVersion: 3.0.0
name: fevm_master_classic_marcus_catalog.rfp_presentation.silver_product_catalog
position:
  x: 1820
  y: 0
description:
  text: Overwrite a table named silver_product_catalog in the given catalog and schema with new data.
  hash: 8141a57a
previewMode: "1000"
config:
  output_type: table
  catalog: fevm_master_classic_marcus_catalog
  schema: rfp_presentation
  table_name: silver_product_catalog
  write_mode: overwrite
input:
  - node: transform_a2b3c4d5
    input_port: data
    output_port: transformed_data
"""

# generated from the system
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_EXCEL_CELLS = 5_000_000

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _file_path(catalog: str, schema: str, volume: str, file_name: str) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        schema_segment = schema.replace(".", "/")
        return f"/Volumes/{catalog}/{schema_segment}/{volume}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _write_single_file(rows, columns, file_type: str, path: str, append: bool) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                else:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        import io

        try:
            import openpyxl
        except ModuleNotFoundError as exc:
            raise ModuleNotFoundError(
                "Writing Excel files requires the 'openpyxl' package, which "
                "is not installed in this environment. Add "
                "openpyxl==3.1.5 to the notebook environment "
                "and apply it, then run again."
            ) from exc

        import pandas as pd
        import re as _re

        try:
            from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
        except ImportError:
            _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

        def _excel_safe(value):
            if isinstance(value, str):
                return _lb_illegal.sub("", value)
            if isinstance(value, (int, float, bool, type(None))):
                return value
            return _lb_illegal.sub("", str(value))

        header = list(columns)
        source_keys = list(columns)
        existing_values: List = []
        if appending:
            workbook = openpyxl.load_workbook(path, read_only=True)
            try:
                sheet = workbook.active
                row_iter = sheet.iter_rows(values_only=True)
                first = next(row_iter, None)
                if first is not None:
                    header = list(first)
                    if sorted(str(c) for c in header) != sorted(
                        str(_excel_safe(c)) for c in columns
                    ):
                        raise ValueError(
                            f"Output: cannot append to {path}: the data's columns "
                            f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                            f"file's columns {[str(c) for c in header]}."
                        )
                    raw_by_safe: Dict[str, Any] = {}
                    for c in columns:
                        raw_by_safe.setdefault(str(_excel_safe(c)), c)
                    source_keys = [raw_by_safe[str(c)] for c in header]
                    row_cap = _excel_row_cap(len(header))
                    for record in row_iter:
                        if len(existing_values) + len(rows) >= row_cap:
                            raise ValueError(
                                f"Output: appending {len(rows):,} rows would grow {path} past "
                                f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                                f"at {len(header):,} columns) — write to a table instead."
                            )
                        existing_values.append(list(record))
            finally:
                workbook.close()

        combined = existing_values + [
            [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
        ]
        new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])

        def _write_excel(tmp_path: str) -> None:
            buffer = io.BytesIO()
            new_df.to_excel(buffer, index=False, engine="openpyxl")
            with open(tmp_path, "wb") as out:
                out.write(buffer.getvalue())

        _stage(_write_excel)
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        _with_extension(config.get("file_name", ""), file_type),
    )
    append = config.get("write_mode", "overwrite") == "append"
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
    else:
        rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
        if len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
    _write_single_file(rows, df.columns, file_type, path, append)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
) -> None:
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        return {}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _run_materialized_view(config, source_view, spark)
        finally:
            spark.catalog.dropTempView(source_view)
        return {}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(stmt, args=args) if args else spark.sql(stmt)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "table",
    "catalog": "fevm_master_classic_marcus_catalog",
    "schema": "rfp_presentation",
    "table_name": "silver_product_catalog",
    "write_mode": "overwrite"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["transform_a2b3c4d5.transformed_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
if not config["meta_state"]["is_disabled"]:
    out = run(config, inputs, spark)
else:
    out = {}